In [8]:
import cv2
import numpy as np
from ultralytics import YOLO
import math
import pandas as pd
from scipy.optimize import linear_sum_assignment

SOURCE_VIDEO_PATH = "../dataset/CarLA/Camera_1/video/camera_1.mp4"
OUTPUT_VIDEO_PATH = "single_camera_tracking_output.mp4"
MODEL_PATH = "yolov8s.pt"

CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5
VEHICLE_CLASSES = [2, 7] 

# Tracker Settings
ASSOCIATION_THRESHOLD = 100
MAX_MISSED_FRAMES = 10
LOST_TRACKER_TIMEOUT = 20   

def data_association(trackers, detections, cost_threshold=50):
    tracker_ids = list(trackers.keys())
    num_trackers = len(tracker_ids)
    num_detections = len(detections)
    if num_trackers == 0 or num_detections == 0:
        return {}, set(tracker_ids), set(range(num_detections))
    cost_matrix = np.zeros((num_trackers, num_detections), dtype=np.float32)
    for i, t_id in enumerate(tracker_ids):
        pred_state = trackers[ t_id ].x  # Use tracker instance state directly
        pred_x, pred_y = pred_state[ 0, 0 ], pred_state[ 1, 0 ]
        for j, detection in enumerate(detections):
            cost_matrix[ i, j ] = math.hypot(pred_x - detection[ "x" ],
                                             pred_y - detection[ "y" ])
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    matched = {}
    unmatched_tracker_ids = set(tracker_ids)
    unmatched_detection_idxs = set(range(num_detections))
    for r, c in zip(row_ind, col_ind):
        if cost_matrix[ r, c ] < cost_threshold:
            t_id = tracker_ids[ r ]
            matched[ t_id ] = detections[ c ]
            unmatched_tracker_ids.discard(t_id)
            unmatched_detection_idxs.discard(c)
    return matched, unmatched_tracker_ids, unmatched_detection_idxs

class EKFTracker:
    def __init__(self, initial_state, P, Q, R, dt):
        self.x = initial_state
        self.P = P
        self.Q = Q
        self.R = R
        self.dt = dt
        self.missed_frames = 0
        self.lost_age = 0
        self.H = np.array([[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 0, 1]])

    def predict(self):
        F = np.eye(5)
        F[0, 2] = self.dt
        F[1, 3] = self.dt
        self.x = F @ self.x
        self.P = F @ self.P @ F.T + self.Q

    def update(self, z):
        z_pred = self.H @ self.x
        y = z - z_pred
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(len(self.x)) - K @ self.H) @ self.P
        self.missed_frames = 0

def data_association(trackers, detections, cost_threshold):
    if not trackers or not detections:
        return {}, set(trackers.keys()), set(range(len(detections)))

    tracker_ids = list(trackers.keys())
    pred_points = np.array([trackers[tid].x[:2, 0] for tid in tracker_ids])
    det_points = np.array([d['point'] for d in detections])
    
    cost_matrix = np.linalg.norm(pred_points[:, np.newaxis, :] - det_points[np.newaxis, :, :], axis=2)
    
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    matched, unmatched_tracker_ids, unmatched_detection_idxs = {}, set(tracker_ids), set(range(len(detections)))
    for r, c in zip(row_ind, col_ind):
        if cost_matrix[r, c] < cost_threshold:
            t_id = tracker_ids[r]
            matched[t_id] = detections[c]
            unmatched_tracker_ids.discard(t_id)
            unmatched_detection_idxs.discard(c)
            
    return matched, unmatched_tracker_ids, unmatched_detection_idxs

In [13]:

model = YOLO("../yolov8s.pt")
cap = cv2.VideoCapture(SOURCE_VIDEO_PATH)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))

results_file = open("single_camera_results.txt", "w")

active_trackers = {}
next_tracker_id = 0
frame_count = 0
dt = 1 / fps if fps > 0 else 1/30

P = np.eye(5) * 50
Q = np.diag([10, 10, 50, 50, 10])
R = np.diag([10, 10, 50])

print("Processing video and saving results...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, classes=VEHICLE_CLASSES, verbose=False)
    
    detections = []
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        center_x = (x1 + x2) // 2
        bottom_y = y2
        detections.append({'point': (center_x, bottom_y), 'bbox': (x1, y1, x2, y2)})

    for tracker in active_trackers.values():
        tracker.predict()

    matched, unmatched_ids, unmatched_dets_indices = data_association(active_trackers, detections, ASSOCIATION_THRESHOLD)

    for tracker_id, detection in matched.items():
        point = detection['point']
        measurement = np.array([point[0], point[1], 0]).reshape(3, 1)
        active_trackers[tracker_id].update(measurement)

    for tracker_id in list(unmatched_ids):
        tracker = active_trackers[tracker_id]
        tracker.missed_frames += 1
        if tracker.missed_frames > MAX_MISSED_FRAMES:
            del active_trackers[tracker_id]
            
    for idx in unmatched_dets_indices:
        detection = detections[idx]
        point = detection['point']
        initial_state = np.array([point[0], point[1], 0, 0, 0]).reshape(5, 1)
        active_trackers[next_tracker_id] = EKFTracker(initial_state, P.copy(), Q.copy(), R.copy(), dt)
        next_tracker_id += 1
    
    for tracker_id, tracker in active_trackers.items():
        state = tracker.x
        pos = (int(state[0, 0]), int(state[1, 0]))
        
        result_line = f"{frame_count},{tracker_id},{pos[0]},{pos[1]},-1,-1,-1,-1\n"
        results_file.write(result_line)

        cv2.putText(frame, str(tracker_id), (pos[0], pos[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.circle(frame, pos, 4, (0, 255, 0), -1)

    out.write(frame)
    frame_count += 1

# Cleanup
cap.release()
out.release()
results_file.close()
cv2.destroyAllWindows()
print(f"Processing complete. Video saved to '{OUTPUT_VIDEO_PATH}' and results saved to 'single_camera_results.txt'")

Processing video and saving results...
Processing complete. Video saved to 'single_camera_tracking_output.mp4' and results saved to 'single_camera_results.txt'


# Evaluate the tracking results against ground-truth 

In [14]:
GT_PATH = "../dataset/CarLA/Camera_1/bboxes.csv"
PREDICTED_PATH = "single_camera_results.txt"
IOU_THRESHOLD = 0.5 

In [15]:

def load_ground_truth(path):
    """Loads and formats the ground truth data."""
    df = pd.read_csv(path)
    df.rename(columns={'Frame': 'FrameId', 'Vehicle_ID': 'Id'}, inplace=True)
    return df

def load_predictions(path, gt_df):
    """Loads predictions and creates bounding boxes using the average GT size."""
    df = pd.read_csv(path, header=None)
    df.columns = ['FrameId', 'Id', 'center_x', 'center_y', 'c5', 'c6', 'c7', 'c8']
    
    avg_width = (gt_df['max_x'] - gt_df['min_x']).mean()
    avg_height = (gt_df['max_y'] - gt_df['min_y']).mean()
    
    df['min_x'] = df['center_x'] - avg_width / 2
    df['min_y'] = df['center_y'] - avg_height
    df['max_x'] = df['center_x'] + avg_width / 2
    df['max_y'] = df['center_y']
    
    return df

def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    
    interArea = max(0, xB - xA) * max(0, yB - yA)
    
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    
    iou = interArea / float(boxAArea + boxBArea - interArea) if (boxAArea + boxBArea - interArea) > 0 else 0
    return iou


In [16]:

try:
    gt_df = load_ground_truth(GT_PATH)
    ts_df = load_predictions(PREDICTED_PATH, gt_df)
    
    gt_by_frame = gt_df.groupby('FrameId')
    ts_by_frame = ts_df.groupby('FrameId')

    matches, misses, false_alarms, id_switches, total_distance = 0, 0, 0, 0, 0.0
    gt_track_history = {}

    max_frame = int(max(gt_df['FrameId'].max(), ts_df['FrameId'].max()))

    for frame_id in range(max_frame + 1):
        # Ensure we always get a DataFrame, even for a single row
        gt_frame = gt_by_frame.get_group(frame_id) if frame_id in gt_by_frame.groups else pd.DataFrame()
        ts_frame = ts_by_frame.get_group(frame_id) if frame_id in ts_by_frame.groups else pd.DataFrame()

        if gt_frame.empty or ts_frame.empty:
            misses += len(gt_frame)
            false_alarms += len(ts_frame)
            continue
            
        gt_boxes = gt_frame[['min_x', 'min_y', 'max_x', 'max_y']].values
        ts_boxes = ts_frame[['min_x', 'min_y', 'max_x', 'max_y']].values

        cost_matrix = np.zeros((len(gt_boxes), len(ts_boxes)))
        for i, gt_box in enumerate(gt_boxes):
            for j, ts_box in enumerate(ts_boxes):
                cost_matrix[i, j] = 1 - calculate_iou(gt_box, ts_box)
        
        gt_ind, ts_ind = linear_sum_assignment(cost_matrix)
        
        matched_gt_indices = set()
        matched_ts_indices = set()

        for r, c in zip(gt_ind, ts_ind):
            if cost_matrix[r, c] < (1 - IOU_THRESHOLD):
                matches += 1
                
                gt_id = gt_frame.iloc[r]['Id']
                ts_id = ts_frame.iloc[c]['Id']
                
                if gt_id in gt_track_history and gt_track_history[gt_id] != ts_id:
                    id_switches += 1
                gt_track_history[gt_id] = ts_id
                
                gt_center = np.array([(gt_boxes[r][0] + gt_boxes[r][2]) / 2, (gt_boxes[r][1] + gt_boxes[r][3]) / 2])
                ts_center = np.array([ts_frame.iloc[c]['center_x'], ts_frame.iloc[c]['center_y']])
                total_distance += np.linalg.norm(gt_center - ts_center)

                matched_gt_indices.add(r)
                matched_ts_indices.add(c)

        misses += len(gt_boxes) - len(matched_gt_indices)
        false_alarms += len(ts_boxes) - len(matched_ts_indices)

    print("\nTracking Evaluation Results:")
    print(f"Matches: {matches}")
    print(f"Misses (False Negatives): {misses}")
    print(f"False Alarms (False Positives): {false_alarms}")
    print(f"ID Switches: {id_switches}")
    if matches > 0:
        motp = total_distance / matches
        print(f"Average Localization Error (MOTP): {motp:.2f} pixels")
    else:
        print("No matches found to calculate localization error.")

except FileNotFoundError as e:
    print(f"\nError: A required file was not found. Please check file paths.\nDetails: {e}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")


Tracking Evaluation Results:
Matches: 1319
Misses (False Negatives): 1753
False Alarms (False Positives): 1495
ID Switches: 3
Average Localization Error (MOTP): 45.17 pixels
